# UniMol aug2 — Inference Only (8-Fold Ensemble)

**Use this notebook when training already completed** and you just need predictions.

**Required Kaggle inputs**:
1. The training notebook's output dataset — contains `unimol_pxr_aug2_8fold/` with `model_0.pth`…`model_7.pth`, `config.yaml`, `target_scaler.ss`
2. A dataset containing `set1_holdout_30_stratified.csv` and `test.csv`

**Outputs** (saved to `/kaggle/working/`):
- `predictions_unimol_aug2_holdout30_raw.csv` — holdout-30 predictions + RAE/MAE evaluation
- `predictions_unimol_aug2_test_raw.csv` — 513-compound test predictions for ensemble blend
- `oof_metrics.txt` — summary metrics

## Cell 1 — Install & GPU Check

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout[:500])
else:
    print('WARNING: No GPU — inference will be slow on CPU')

!pip install -q unimol-tools huggingface_hub

import torch
print(f'PyTorch {torch.__version__}  CUDA={torch.cuda.is_available()}')

## Cell 2 — Configuration & Locate Files

In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

SMILES_COL = 'SMILES'
TARGET_COL = 'pEC50'
NAME_COL   = 'Molecule Name'

def find_file(filename):
    """Search /kaggle/input recursively for a file by name."""
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f'{filename} not found under /kaggle/input')

def find_dir(dirname):
    """Search /kaggle/input recursively for a directory by name."""
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if d == dirname:
                return os.path.join(root, d)
    raise FileNotFoundError(f'Directory {dirname!r} not found under /kaggle/input')

# ── Locate saved model directory ──────────────────────────────────────────────
MODEL_DIR = find_dir('unimol_pxr_aug2_8fold')

# Verify required files are present
model_files = sorted(glob.glob(os.path.join(MODEL_DIR, 'model_*.pth')))
config_file = os.path.join(MODEL_DIR, 'config.yaml')
scaler_file = os.path.join(MODEL_DIR, 'target_scaler.ss')

assert len(model_files) > 0, f'No model_*.pth files found in {MODEL_DIR}'
assert os.path.exists(config_file), f'config.yaml not found in {MODEL_DIR}'
assert os.path.exists(scaler_file), f'target_scaler.ss not found in {MODEL_DIR}'

print(f'Model dir  : {MODEL_DIR}')
print(f'Models     : {[os.path.basename(m) for m in model_files]}')
print(f'config.yaml: OK')
print(f'target_scaler.ss: OK')

# ── Locate input CSVs ────────────────────────────────────────────────────────
HOLDOUT_CSV = find_file('set1_holdout_30_stratified.csv')
TEST_CSV    = find_file('test.csv')
print(f'\nHoldout CSV: {HOLDOUT_CSV}')
print(f'Test CSV   : {TEST_CSV}')

# ── Conformer generation settings ───────────────────────────────────────────
CONF_SEED         = 42
CONF_MAX_ATTEMPTS = 5
REMOVE_HS         = False

# ── Output paths ─────────────────────────────────────────────────────────────
HOLDOUT_OUT_CSV  = '/kaggle/working/predictions_unimol_aug2_holdout30_raw.csv'
TEST_OUT_RAW_CSV = '/kaggle/working/predictions_unimol_aug2_test_raw.csv'
METRICS_TXT      = '/kaggle/working/oof_metrics.txt'

print(f'\nAll checks passed — ready for inference.')

## Cell 3 — Load & Validate Data

In [ ]:
def load_and_validate(path, require_target=True):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for col in [SMILES_COL, TARGET_COL, NAME_COL]:
        if col not in df.columns:
            match = next((c for c in df.columns if c.lower() == col.lower()), None)
            if match:
                df.rename(columns={match: col}, inplace=True)
    if SMILES_COL not in df.columns:
        raise ValueError(f"'{SMILES_COL}' not found. Columns: {list(df.columns)}")
    if require_target and TARGET_COL not in df.columns:
        raise ValueError(f"'{TARGET_COL}' not found.")
    cols = [SMILES_COL, TARGET_COL] if require_target else [SMILES_COL]
    before = len(df)
    df = df.dropna(subset=cols).reset_index(drop=True)
    if len(df) < before:
        print(f'  Dropped {before-len(df)} rows with missing values')
    return df

holdout_df = load_and_validate(HOLDOUT_CSV, require_target=True)
test_df    = load_and_validate(TEST_CSV,    require_target=False)

y_holdout  = holdout_df[TARGET_COL].values.astype(np.float32)

print(f'Holdout : N={len(holdout_df)}  pEC50=[{y_holdout.min():.2f}, {y_holdout.max():.2f}]  mean={y_holdout.mean():.3f}')
print(f'          <4.0={( y_holdout<4.0).sum()}  4-5.5={(( y_holdout>=4.0)&(y_holdout<5.5)).sum()}  >=5.5={(y_holdout>=5.5).sum()}')
print(f'Test    : N={len(test_df)}')

## Cell 4 — Generate 3D Conformers (Holdout + Test)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

def generate_conformers(smiles_list, seed=42, max_attempts=5, remove_hs=False, label=''):
    atoms_list, coords_list, failed = [], [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            atoms_list.append(None); coords_list.append(None); failed.append(i)
            continue
        mol_h = Chem.AddHs(mol)
        ok = -1
        for attempt in range(max_attempts):
            p = AllChem.ETKDGv3()
            p.randomSeed = seed + attempt
            p.numThreads = 0
            p.enforceChirality = True
            p.useSmallRingTorsions = True
            ok = AllChem.EmbedMolecule(mol_h, p)
            if ok == 0:
                break
        if ok != 0:
            ok = AllChem.EmbedMolecule(mol_h, AllChem.ETKDG())
        if ok != 0:
            atoms_list.append(None); coords_list.append(None); failed.append(i)
            continue
        try:
            AllChem.MMFFOptimizeMolecule(mol_h, maxIters=2000)
        except Exception:
            pass
        if remove_hs:
            mol_h = Chem.RemoveHs(mol_h)
        atoms_list.append([a.GetSymbol() for a in mol_h.GetAtoms()])
        coords_list.append(mol_h.GetConformer().GetPositions().astype(np.float64))
    print(f'{label}: {len(smiles_list)} molecules  |  conformer failures: {len(failed)}')
    return atoms_list, coords_list, failed

t0 = time.time()
holdout_atoms, holdout_coords, holdout_failed = generate_conformers(
    holdout_df[SMILES_COL].tolist(), seed=CONF_SEED,
    max_attempts=CONF_MAX_ATTEMPTS, remove_hs=REMOVE_HS, label='Holdout-30')
test_atoms, test_coords, test_failed = generate_conformers(
    test_df[SMILES_COL].tolist(),    seed=CONF_SEED,
    max_attempts=CONF_MAX_ATTEMPTS, remove_hs=REMOVE_HS, label='Test (Set 2)')
print(f'Conformer generation done in {(time.time()-t0)/60:.1f} min')

## Cell 5 — Predict Holdout-30 (Validation)

Full 8-fold ensemble prediction using all saved checkpoints.

In [ ]:
from unimol_tools import MolPredict

def predict_ensemble(df, atoms, coords, model_dir, label):
    """Run full ensemble prediction from model_dir. Returns predictions aligned to df."""
    n = len(df)
    data = {
        'SMILES'      : df[SMILES_COL].tolist(),
        'atoms'       : atoms,
        'coordinates' : coords,
        TARGET_COL    : [0.0] * n,
    }
    print(f'Predicting {n} {label} compounds using full ensemble...')
    t0 = time.time()
    predictor = MolPredict(load_model=model_dir)
    raw = np.array(predictor.predict(data=data)).flatten()
    # Guard: if MolPredict silently drops failed conformers, pad with training mean
    if len(raw) < n:
        print(f'  WARNING: received {len(raw)} predictions for {n} inputs — padding gaps')
        preds     = np.full(n, float(np.nanmean(raw)))
        valid_idx = [i for i, a in enumerate(atoms) if a is not None]
        for k, vi in enumerate(valid_idx[:len(raw)]):
            preds[vi] = raw[k]
    else:
        preds = raw
    elapsed = (time.time() - t0) / 60
    print(f'  Done in {elapsed:.1f} min')
    print(f'  range=[{preds.min():.3f}, {preds.max():.3f}]  '
          f'mean={preds.mean():.3f}  std={preds.std():.3f}')
    return preds

def rae(y, yhat):
    return float(np.sum(np.abs(y - yhat)) / np.sum(np.abs(y - y.mean())))

# ── Run holdout inference ─────────────────────────────────────────────────────
holdout_preds = predict_ensemble(holdout_df, holdout_atoms, holdout_coords, MODEL_DIR, 'holdout-30')

# ── Evaluate ──────────────────────────────────────────────────────────────────
holdout_rae  = rae(y_holdout, holdout_preds)
holdout_mae  = float(np.mean(np.abs(y_holdout - holdout_preds)))
holdout_sp   = float(spearmanr(y_holdout, holdout_preds).statistic)
holdout_bias = float((holdout_preds - y_holdout).mean())

print(f'\n{"="*55}')
print('HOLDOUT-30 RESULTS (honest out-of-sample)')
print(f'  RAE      = {holdout_rae:.4f}')
print(f'  MAE      = {holdout_mae:.4f}')
print(f'  Spearman = {holdout_sp:.4f}')
print(f'  Bias     = {holdout_bias:+.4f}')
print()
print('  Per-activity-region:')
for lo, hi, lbl in [(-99, 4.0, 'Inactive   <4.0'),
                    (4.0, 5.5, 'Moderate 4-5.5'),
                    (5.5, 99,  'Potent    >5.5 ')]:
    mask = (y_holdout > lo) & (y_holdout <= hi)
    if mask.any():
        b = float((holdout_preds - y_holdout)[mask].mean())
        m = float(np.abs(holdout_preds - y_holdout)[mask].mean())
        print(f'    {lbl}: n={mask.sum():2d}  bias={b:+.3f}  MAE={m:.3f}')
print(f'{"="*55}')

## Cell 6 — Predict Test Set (Submission)

In [ ]:
test_preds = predict_ensemble(test_df, test_atoms, test_coords, MODEL_DIR, 'test Set 2')

print(f'\nTest predictions summary:')
print(f'  N         = {len(test_preds)}')
print(f'  range     = [{test_preds.min():.3f}, {test_preds.max():.3f}]')
print(f'  mean/std  = {test_preds.mean():.3f} / {test_preds.std():.3f}')
print(f'  >5.5      = {(test_preds>5.5).sum()}')
print(f'  >6.0      = {(test_preds>6.0).sum()}')

## Cell 7 — Save Output Files

Download these from the Output tab when the notebook finishes.

In [ ]:
# ── Holdout-30: predictions + true values ─────────────────────────────────────
holdout_out = holdout_df[[NAME_COL, SMILES_COL, TARGET_COL]].copy()
holdout_out['pEC50_pred'] = holdout_preds
holdout_out['error']      = holdout_preds - holdout_out[TARGET_COL]
holdout_out['abs_error']  = np.abs(holdout_out['error'])
holdout_out = holdout_out.sort_values('abs_error', ascending=False)
holdout_out.to_csv(HOLDOUT_OUT_CSV, index=False)
print(f'Saved: {HOLDOUT_OUT_CSV}')
print(f'  RAE={holdout_rae:.4f}  MAE={holdout_mae:.4f}  Spearman={holdout_sp:.4f}  N={len(holdout_out)}')

print()

# ── Test Set 2: predictions for blending ──────────────────────────────────────
name_col_test = next((c for c in test_df.columns
                      if c.lower().replace(' ', '_') in ('molecule_name', 'id')), None)
test_out = pd.DataFrame()
if name_col_test:
    test_out[NAME_COL] = test_df[name_col_test].values
else:
    test_out[NAME_COL] = [f'compound_{i+1}' for i in range(len(test_df))]
test_out[SMILES_COL] = test_df[SMILES_COL].values
test_out['pEC50']    = test_preds
test_out.to_csv(TEST_OUT_RAW_CSV, index=False)
print(f'Saved: {TEST_OUT_RAW_CSV}')
print(f'  range=[{test_preds.min():.3f}, {test_preds.max():.3f}]  mean={test_preds.mean():.3f}  N={len(test_out)}')

print()

# ── Metrics summary text file ─────────────────────────────────────────────────
summary = [
    'UniMol aug2 — 8-Fold Ensemble Inference',
    f'Model dir : {MODEL_DIR}',
    f'Folds     : {len(model_files)}',
    '',
    'HOLDOUT-30 (honest out-of-sample):',
    f'  RAE      = {holdout_rae:.4f}',
    f'  MAE      = {holdout_mae:.4f}',
    f'  Spearman = {holdout_sp:.4f}',
    f'  Bias     = {holdout_bias:+.4f}',
    '',
    'TEST SET 2:',
    f'  N        = {len(test_preds)}',
    f'  range    = [{test_preds.min():.3f}, {test_preds.max():.3f}]',
    f'  mean/std = {test_preds.mean():.3f} / {test_preds.std():.3f}',
    f'  >5.5     = {(test_preds>5.5).sum()}',
    f'  >6.0     = {(test_preds>6.0).sum()}',
]
with open(METRICS_TXT, 'w') as f:
    f.write('\n'.join(summary))
print(f'Saved: {METRICS_TXT}')

print(f'\n{"="*55}')
print('ALL DONE — download these files from Output tab:')
print(f'  predictions_unimol_aug2_holdout30_raw.csv')
print(f'  predictions_unimol_aug2_test_raw.csv')
print(f'  oof_metrics.txt')
print(f'{"="*55}')